<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/toy_model/NotWorking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd

In [1]:
!pip install transformer_lens

In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm

E = 100  # num entities
T = 10   # num types/relations

SEP = E + T
Q = E + T + 1
PAD = E + T + 2
D_VOCAB = E + T + 3

ENTITIES = np.arange(0, E)
TYPES    = np.arange(E, E + T)

N_WORLDS = 80_000
MIN_FACTS, MAX_FACTS = 4, 8
SEED = 0

rng = np.random.default_rng(SEED)

def produce_example(num_relations: int, *, allow_self_loops: bool = False):
    facts = []
    seen_head_rel = set()

    while len(facts) < num_relations:
        e = int(rng.integers(0, E))
        t = int(TYPES[rng.integers(0, T)])

        if (e, t) in seen_head_rel:
            continue  # reject duplicate (e, t) - because then our question (t, e) would no longer be deterministic

        # forbid self-loop (can discuss?)
        e2 = int(rng.integers(0, E))
        while e2 == e:
            e2 = int(rng.integers(0, E))

        seen_head_rel.add((e, t))
        facts.append((e, t, e2))

    # choose a query by selecting one of the world facts
    q_idx = int(rng.integers(0, num_relations))
    Eq, Tq, E2q = facts[q_idx]

    seq = []
    for (e, t, e2) in facts:
        seq.extend([e, t, e2, SEP])
    seq.extend([Tq, Eq, Q])

    label = E2q
    return seq, label, facts

# --- Build dataset ---
rows = []
for _ in tqdm(range(N_WORLDS)):
    k = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))
    seq, label, _ = produce_example(k, allow_self_loops=False)
    rows.append({"tokens": seq, "label": label})

df = pd.DataFrame(rows)

100%|██████████| 80000/80000 [00:05<00:00, 15384.31it/s]


In [4]:
df

,tokens,label
0,"[63, 105, 26, 110, 30, 100, 7, 110, 1, 101, 81...",26
1,"[72, 108, 17, 110, 8, 108, 2, 110, 54, 100, 29...",52
2,"[46, 109, 80, 110, 98, 103, 68, 110, 95, 106, ...",38
3,"[52, 103, 31, 110, 42, 104, 71, 110, 88, 100, ...",71
4,"[62, 100, 8, 110, 37, 108, 40, 110, 78, 103, 2...",33
...,...,...
79995,"[26, 100, 22, 110, 7, 107, 13, 110, 21, 106, 6...",86
79996,"[71, 102, 9, 110, 87, 106, 54, 110, 43, 101, 6...",68
79997,"[32, 104, 40, 110, 22, 105, 96, 110, 97, 104, ...",40
79998,"[80, 105, 23, 110, 93, 105, 5, 110, 20, 101, 9...",23


In [5]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split

In [6]:
IGNORE_INDEX = -100

class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

In [7]:
def train_collate_fn(batch, rng=np.random.default_rng()):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = shuffle_facts(seq.tolist() if torch.is_tensor(seq) else seq, rng)
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target

def val_collate_fn(batch):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = seq.tolist() if torch.is_tensor(seq) else seq
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target


In [8]:
def shuffle_facts(seq, rng=np.random.default_rng()):
    seps = [i for i,t in enumerate(seq) if t == SEP]
    context = seq[:seps[-1]+1]
    query_part = seq[seps[-1]+1:]
    facts = [context[i:i+4] for i in range(0, len(context), 4)]
    rng.shuffle(facts)

    return [x for f in facts for x in f] + query_part

In [9]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_dataset = EntityBindingDataset(train_df)
val_dataset = EntityBindingDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=train_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=val_collate_fn)

# Save dataframes
train_df.to_csv("train_df.csv", index=False)
val_df.to_csv("val_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)

In [10]:
len(train_dataset)

64000

In [11]:
def compute_accuracy(logits: torch.Tensor, targets: torch.Tensor, ignore_index: int = IGNORE_INDEX):
    """
    Accuracy over positions where targets != ignore_index.
    Returns correct_count and total_count
    """
    with torch.no_grad():
        mask = targets.ne(ignore_index)
        total = mask.sum().item()
        preds = logits.argmax(dim=-1)
        correct = preds.masked_select(mask).eq(targets.masked_select(mask)).sum().item()
        return correct, total

In [23]:
import math
import itertools
import torch
import torch.nn as nn
from transformer_lens import HookedTransformer, HookedTransformerConfig
from torch.utils.tensorboard import SummaryWriter

# hparams

LAYERS = [3]
HEADS  = [1]

d_model = 32
d_head = 32
n_ctx = 64
lr = 3e-4
betas = (0.9, 0.98)
weight_decay = 0.01
num_epochs = 30

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        act_fn="gelu",
        attn_only=True,
        normalization_type="LN",
    )
    return HookedTransformer(cfg)

In [24]:
def grid_search(n_layers: int, n_heads: int, model=None):
    # Fresh seeds per run for comparability
    torch.manual_seed(0)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(0)

    if model is None:
      model = build_model(n_layers, n_heads)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=betas, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    run_name = f"entity_binding_L{n_layers}_H{n_heads}"
    writer = SummaryWriter(f"runs/{run_name}")

    global_step = 0
    best_val_acc = -1.0
    best_val_epoch = -1

    for epoch in range(100):
        model.train()
        total_train_loss = 0
        total_train_correct = 0
        total_train_count = 0
        for input_tokens, targets in train_loader:
            input_tokens, targets = input_tokens.to(device), targets.to(device)

            optimizer.zero_grad()
            logits = model(input_tokens)  # shape: [B, T, d_vocab]
            loss = criterion(logits.view(-1, logits.size(-1)),
                              targets.view(-1))
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            c, n = compute_accuracy(logits, targets)
            total_train_correct += c
            total_train_count += n
            step_acc = (c / n) if n else 0.0
            writer.add_scalar("Loss/train", loss.item(), global_step)
            writer.add_scalar("Acc/train_step", step_acc, global_step)
            global_step += 1

        avg_train_loss = total_train_loss / len(train_loader)
        train_acc = (total_train_correct / total_train_count) if total_train_count else 0.0
        print(f"[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.3f}")

        model.eval()
        total_val_loss = 0
        total_val_correct = 0
        total_val_count = 0
        with torch.no_grad():
            for input_tokens, targets in val_loader:
                input_tokens, targets = input_tokens.to(device), targets.to(device)
                logits = model(input_tokens)
                loss = criterion(logits.view(-1, logits.size(-1)),
                              targets.view(-1))
                total_val_loss += loss.item()
                c, n = compute_accuracy(logits, targets)
                total_val_correct += c
                total_val_count += n

        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = (total_val_correct / total_val_count) if total_val_count else 0.0
        print(f"[Epoch {epoch+1}] Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.3f}")
        writer.add_scalar("Loss/val", avg_val_loss, global_step)
        writer.add_scalar("Acc/val", val_acc, global_step)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_epoch = epoch

    writer.close()
    print("Training complete.")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model

In [ ]:
results = []
for n_layers, n_heads in itertools.product(LAYERS, HEADS):
    print(f"##### Running config: n_layers={n_layers}, n_heads={n_heads} #####")
    model = grid_search(n_layers, n_heads)

##### Running config: n_layers=3, n_heads=1 #####
Moving model to device:  cuda
[Epoch 1] Train Loss: 4.3164 | Train Acc: 0.066
[Epoch 1] Val Loss: 3.8377 | Val Acc: 0.129
[Epoch 2] Train Loss: 3.5675 | Train Acc: 0.155
[Epoch 2] Val Loss: 3.3717 | Val Acc: 0.157
[Epoch 3] Train Loss: 3.2193 | Train Acc: 0.177
[Epoch 3] Val Loss: 3.1512 | Val Acc: 0.170
[Epoch 4] Train Loss: 3.0325 | Train Acc: 0.184
[Epoch 4] Val Loss: 3.0162 | Val Acc: 0.172
[Epoch 5] Train Loss: 2.9164 | Train Acc: 0.191
[Epoch 5] Val Loss: 2.9369 | Val Acc: 0.178
[Epoch 6] Train Loss: 2.8306 | Train Acc: 0.199
[Epoch 6] Val Loss: 2.8819 | Val Acc: 0.182
[Epoch 7] Train Loss: 2.7615 | Train Acc: 0.204
[Epoch 7] Val Loss: 2.8280 | Val Acc: 0.182
[Epoch 8] Train Loss: 2.7113 | Train Acc: 0.208
[Epoch 8] Val Loss: 2.7864 | Val Acc: 0.185
[Epoch 9] Train Loss: 2.6710 | Train Acc: 0.209
[Epoch 9] Val Loss: 2.7539 | Val Acc: 0.187
[Epoch 10] Train Loss: 2.6399 | Train Acc: 0.211
[Epoch 10] Val Loss: 2.7378 | Val Acc: 0.18

In [ ]:
# Save the model
torch.save(model.state_dict(), "entity_binding_model.pth")

# Load the model
# Create a new model instance with the same configuration
loaded_model = HookedTransformer(cfg)
loaded_model.load_state_dict(torch.load("entity_binding_model.pth"))
loaded_model = loaded_model.to(device)

print("Model saved and loaded successfully.")

In [ ]:

loaded_model.eval()
total_val_loss = 0
total_val_correct = 0
total_val_count = 0
with torch.no_grad():
    for input_tokens, targets in val_loader:
        input_tokens, targets = input_tokens.to(device), targets.to(device)
        logits = loaded_model(input_tokens)
        loss = criterion(logits.view(-1, logits.size(-1)),
                      targets.view(-1))
        total_val_loss += loss.item()
        c, n = compute_accuracy(logits, targets)
        total_val_correct += c
        total_val_count += n

avg_val_loss = total_val_loss / len(val_loader)
val_acc = (total_val_correct / total_val_count) if total_val_count else 0.0

In [ ]:
val_acc, avg_val_loss